Copyright 2026 Snowflake Inc.
SPDX-License-Identifier: Apache-2.0

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

# Exercise 2.1: Moving Existing Tables to Iceberg

Migrate existing data into Iceberg using four strategies:
- **CTAS** (Create Table As Select) — reserializes data; works for any format
- **`snapshot` procedure** — metadata-only over existing Parquet (no data rewrite). *Migration tool — not the same as table snapshots that track version history.*
- **`add_files` procedure** — incrementally register Parquet files without rewriting
- **Metadata tables** — inspect migrated structure

You'll benchmark CTAS vs snapshot, and `add_files` vs `INSERT INTO`.

⚠️ **Spark Connect**: one Spark server in the background, notebooks are thin clients. If `ConnectionRefusedError` appears, check `docker logs jupyter-spark` or restart with `docker compose restart jupyter`.

## Initialize Spark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import urllib.request
import os
import time

spark = SparkSession.builder \
    .appName("MovingExistingTables") \
    .getOrCreate()

print(f"Spark {spark.version} initialized!")

## Create Namespace

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS polaris.migration")
print("Namespace 'migration' created!")

## Download NYC Taxi Data

Yellow Taxi trips, **June–October 2023** (~225 MB). Enough volume to see real differences between migration strategies.

In [ ]:
import boto3
from botocore.client import Config

# Configure MinIO client using S3-compatible API
s3_client = boto3.client(
    's3',
    endpoint_url='http://minio:9000',
    aws_access_key_id='admin',
    aws_secret_access_key='password',
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

# Download NYC Yellow Taxi data for June through October 2023 and upload to MinIO
base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-{:02d}.parquet"
bucket_name = "warehouse"
minio_prefix = "raw"
temp_dir = "/tmp/nyc_taxi_download"

# Create temp directory
os.makedirs(temp_dir, exist_ok=True)

# Download data for months 6-10 (June through October)
months = range(6, 11)
uploaded_files = []

for month in months:
    url = base_url.format(month)
    filename = f"yellow_tripdata_2023-{month:02d}.parquet"
    local_path = os.path.join(temp_dir, filename)
    minio_key = f"{minio_prefix}/{filename}"
    
    try:
        # Check if file already exists locally
        if os.path.exists(local_path):
            print(f"{filename} already exists locally, skipping download")
        else:
            # Download file
            print(f"Downloading {filename} (~45MB)...")
            urllib.request.urlretrieve(url, local_path)
            print(f"  Downloaded to {local_path}")
        
        # Upload to MinIO
        print(f"  Uploading to MinIO: s3a://{bucket_name}/{minio_key}...")
        s3_client.upload_file(local_path, bucket_name, minio_key)
        print(f"  Uploaded successfully!")
        uploaded_files.append(minio_key)
        
        # Clean up local file
        os.remove(local_path)
    except Exception as e:
        print(f"  Error processing {filename}: {e}")

print(f"\nAll files ready in MinIO! Total: {len(uploaded_files)} months of data")
print(f"Location: s3a://{bucket_name}/{minio_prefix}/")

## Part 1: Migrate CSV Data Using CTAS

CTAS is the most flexible approach — works with any source and allows transformation during migration.

### Create Sample CSV Data

In [ ]:
# Create a sample CSV file and upload to MinIO
csv_data = """customer_id,customer_name,signup_date,total_purchases
1,Alice Johnson,2023-01-15,45
2,Bob Smith,2023-02-20,12
3,Carol White,2023-03-10,78
4,David Brown,2023-04-05,34
5,Eve Davis,2023-05-12,56"""

# Write to temp file, then upload to MinIO
local_csv = '/tmp/customers.csv'
with open(local_csv, 'w') as f:
    f.write(csv_data)

csv_s3_key = 'migration/csv_landing/customers.csv'
s3_client.upload_file(local_csv, 'warehouse', csv_s3_key)
os.remove(local_csv)

print(f"CSV file uploaded to s3://warehouse/{csv_s3_key}")

### Read CSV File

[Open `csv_landing` in MinIO](http://localhost:9001/browser/warehouse/migration/csv_landing/) · `admin / password`

In [ ]:
# Read CSV file from MinIO
csv_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("s3a://warehouse/migration/csv_landing/customers.csv")

# Show schema
print("CSV Schema:")
csv_df.printSchema()

# Show data
print("\nCSV Data:")
csv_df.show()

### Migrate to Iceberg

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.migration.customers_from_csv")

csv_df.writeTo("polaris.migration.customers_from_csv") \
    .using("iceberg") \
    .create()

print("Table created from CSV!")

In [ ]:
# Query the new Iceberg table
spark.sql("SELECT * FROM polaris.migration.customers_from_csv ORDER BY customer_id").show()

**What happened?** Spark read the CSV, parsed it, and wrote new Parquet files (full reserialization). Original CSV untouched. Most flexible, but most expensive.

### View Iceberg Metadata

In [ ]:
# View files in the table
spark.sql("""
SELECT 
    file_path,
    file_size_in_bytes,
    record_count
FROM polaris.migration.customers_from_csv.files
""").show(truncate=False)

### View in MinIO

[Open `customers_from_csv` in MinIO](http://localhost:9001/browser/warehouse/migration/customers_from_csv/) · `admin / password`

`data/` → Parquet files, `metadata/` → Iceberg metadata.

### Try It: Migrate Your Own CSV

Define CSV inline (Option A) or drop a file into `data/jupyter/` (Option B, read via `file:///data/jupyter/`). The `data/jupyter/` host folder is mounted into the container at `/data/jupyter/`.

In [ ]:
# my_table_name = "polaris.migration.???"

# Option A: Define CSV inline and upload to MinIO
# my_csv = """???"""
# with open('/tmp/my_data.csv', 'w') as f:
#     f.write(my_csv)
# s3_client.upload_file('/tmp/my_data.csv', 'warehouse', 'migration/csv_landing/my_data.csv')
# os.remove('/tmp/my_data.csv')
# my_df = spark.read.option("header", "true").option("inferSchema", "true") \
#     .csv("s3a://warehouse/migration/csv_landing/my_data.csv")

# Option B: Place a CSV in data/jupyter/ and read it directly
# my_df = spark.read.option("header", "true").option("inferSchema", "true") \
#     .csv("file:///data/jupyter/???.csv")

# Inspect
# my_df.printSchema()
# my_df.show()

# Migrate to Iceberg using CTAS
# spark.sql(f"DROP TABLE IF EXISTS {my_table_name}")
# my_df.writeTo(my_table_name).using("iceberg").create()

# Verify
# spark.sql(f"SELECT * FROM {my_table_name}").show()

## Part 2: Migrate Parquet Data Using CTAS

CTAS over Parquet re-serializes — new Parquet files managed by Iceberg.

### Examine the Parquet Data

In [ ]:
# Read the June Parquet file from our downloaded data
parquet_path = "s3a://warehouse/raw/yellow_tripdata_2023-06.parquet"
parquet_df = spark.read.parquet(parquet_path)

print("Schema:")
parquet_df.printSchema()

print(f"\nRow count: {parquet_df.count():,}")

print("\nSample data:")
parquet_df.show(5)

### Migrate Parquet to Iceberg Using CTAS

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.migration.taxi_from_ctas")

print("Creating Iceberg table from Parquet using CTAS...")

ctas_start = time.time()

spark.sql("""
CREATE TABLE polaris.migration.taxi_from_ctas
USING iceberg
AS SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-06.parquet`
""")

ctas_time = time.time() - ctas_start
print(f"CTAS table created in {ctas_time:.2f}s")

In [ ]:
# Verify row count
iceberg_count = spark.sql("""
SELECT COUNT(*) as count 
FROM polaris.migration.taxi_from_ctas
""").collect()[0]['count']

print(f"Iceberg table row count: {iceberg_count:,}")

### View in MinIO

[Open `taxi_from_ctas` in MinIO](http://localhost:9001/browser/warehouse/migration/taxi_from_ctas/) · `admin / password`

New Parquet files appear in the Iceberg warehouse, separate from `raw/`. CTAS always makes a full copy.

## Part 3: Migrate Parquet Using `snapshot`

`snapshot` builds Iceberg metadata **over existing Parquet files in-place** — no data rewrite, metadata only. Much faster than CTAS.

Ideal when data is already Parquet, you want the fastest migration, and no transform/repartition is needed.

In [ ]:
# Helper: drop an Iceberg table via the Polaris REST API
# This works even if the table's metadata files are missing in storage
import urllib.request as _req
import json as _json

def polaris_drop_table(namespace, table):
    token_url = "http://polaris:8181/api/catalog/v1/oauth/tokens"
    data = "grant_type=client_credentials&client_id=root&client_secret=s3cr3t&scope=PRINCIPAL_ROLE:ALL"
    req = _req.Request(token_url, data=data.encode(), method="POST",
                       headers={"Content-Type": "application/x-www-form-urlencoded"})
    token = _json.loads(_req.urlopen(req).read())["access_token"]

    drop_url = f"http://polaris:8181/api/catalog/v1/quickstart_catalog/namespaces/{namespace}/tables/{table}"
    req = _req.Request(drop_url, method="DELETE",
                       headers={"Authorization": f"Bearer {token}"})
    try:
        _req.urlopen(req)
    except Exception:
        pass

polaris_drop_table("migration", "taxi_from_snapshot")

# Clean up any previous snapshot location, then copy June data
snapshot_prefix = 'migration/taxi_snapshot/'
bucket = 'warehouse'

existing = s3_client.list_objects_v2(Bucket=bucket, Prefix=snapshot_prefix)
if 'Contents' in existing:
    for obj in existing['Contents']:
        s3_client.delete_object(Bucket=bucket, Key=obj['Key'])
    print(f"Cleaned up {len(existing['Contents'])} existing files from {snapshot_prefix}")

# Copy June data to a dedicated location for the snapshot table
snapshot_data_path = f'{snapshot_prefix}data/'

print("Copying June parquet file to snapshot location...")
s3_client.copy_object(
    Bucket=bucket,
    CopySource=f'{bucket}/raw/yellow_tripdata_2023-06.parquet',
    Key=f'{snapshot_data_path}yellow_tripdata_2023-06.parquet'
)
print(f"File copied to s3://{bucket}/{snapshot_data_path}")

### View Raw Data Files Before Snapshot

[Open `taxi_snapshot` in MinIO](http://localhost:9001/browser/warehouse/migration/taxi_snapshot/) · `admin / password`

Before: only `data/` with raw Parquet. After: a `metadata/` directory appears.

In [ ]:
# The snapshot procedure expects a registered table as input, not a raw file path.
# Spark always has a built-in catalog called spark_catalog for non-Iceberg tables.
# You can have multiple catalogs configured simultaneously and switch between them
# with USE. We switch to spark_catalog here because the Polaris catalog only
# manages Iceberg tables. We need spark_catalog to register a temporary Parquet
# table. After the snapshot, we switch back to polaris.
spark.sql("USE spark_catalog")

spark.sql("DROP TABLE IF EXISTS default.temp_parquet_source")

spark.sql("""
    CREATE TABLE default.temp_parquet_source
    USING parquet
    LOCATION 's3a://warehouse/migration/taxi_snapshot/data/'
""")

# Run the snapshot procedure: creates Iceberg metadata without rewriting data
print("Running snapshot procedure...")
snapshot_start = time.time()

spark.sql("""
    CALL polaris.system.snapshot(
        source_table => 'default.temp_parquet_source',
        table => 'polaris.migration.taxi_from_snapshot',
        location => 's3a://warehouse/migration/taxi_snapshot'
    )
""").show()

snapshot_time = time.time() - snapshot_start
print(f"Snapshot completed in {snapshot_time:.2f}s")

# Switch back to polaris catalog
spark.sql("USE polaris")

# Clean up temporary table
spark.sql("DROP TABLE IF EXISTS spark_catalog.default.temp_parquet_source")

In [ ]:
# Verify the snapshot table
spark.sql("""
    SELECT 
        COUNT(*) as total_trips,
        MIN(tpep_pickup_datetime) as first_pickup,
        MAX(tpep_pickup_datetime) as last_pickup
    FROM polaris.migration.taxi_from_snapshot
""").show()

In [ ]:
print("Snapshot table files:")
spark.sql("""
    SELECT 
        file_path,
        ROUND(file_size_in_bytes / 1024 / 1024, 2) as size_mb,
        record_count
    FROM polaris.migration.taxi_from_snapshot.files
""").show(truncate=False)

### View Snapshot Table in MinIO

[Open `taxi_snapshot` in MinIO](http://localhost:9001/browser/warehouse/migration/taxi_snapshot/) · `admin / password`

`metadata/` now sits beside `data/`. Original data files were **not** rewritten — Iceberg just references them.

### Try It: Inspect the Metadata

Compare metadata tables of CTAS vs snapshot. Look at `file_path` in `.files` (CTAS → warehouse; snapshot → original location) and the `manifests` table (file/manifest counts).

In [ ]:
# snapshotted_table = "polaris.migration.taxi_from_snapshot"
# ctas_table = "polaris.migration.taxi_from_ctas"

# Explore metadata using: spark.sql("SELECT * FROM $table.$metadata_table").show(truncate=False)
# Available metadata tables: snapshots, files, manifests, history, partitions
# Start with the `files` metadata table. Look at the `file_path` column to see where each approach stored data.

# Check snapshots from both tables
# spark.sql(f"SELECT * FROM {snapshotted_table}.snapshots").show(truncate=False)
# spark.sql(f"SELECT * FROM {ctas_table}.snapshots").show(truncate=False)

# Compare the data files. Look at the file_path column
# spark.sql(f"SELECT file_path, record_count FROM {snapshotted_table}.files").show(truncate=False)
# spark.sql(f"SELECT file_path, record_count FROM {ctas_table}.files").show(truncate=False)

# Compare manifest counts. How many manifests does each table have?
# spark.sql(f"SELECT path, added_data_files_count FROM {snapshotted_table}.manifests").show(truncate=False)
# spark.sql(f"SELECT path, added_data_files_count FROM {ctas_table}.manifests").show(truncate=False)

### Benchmark: CTAS vs Snapshot

Compare migration time for the same June 2023 Parquet data.

In [ ]:
print("=" * 60)
print("Migration Benchmark: CTAS vs Snapshot")
print("=" * 60)
print(f"  CTAS time:     {ctas_time:>8.2f}s  (full data reserialization)")
print(f"  Snapshot time: {snapshot_time:>8.2f}s  (metadata-only)")
print(f"  Speedup:       {ctas_time / snapshot_time:>8.1f}x faster with Snapshot")
print("=" * 60)

ctas_count = spark.sql("SELECT COUNT(*) FROM polaris.migration.taxi_from_ctas").collect()[0][0]
snapshot_count = spark.sql("SELECT COUNT(*) FROM polaris.migration.taxi_from_snapshot").collect()[0][0]

print(f"\n  CTAS row count:     {ctas_count:,}")
print(f"  Snapshot row count: {snapshot_count:,}")
print(f"  Both tables have identical data!")

## Part 4: Incremental Data Addition

`add_files` registers existing Parquet files into an Iceberg table **without rewriting** — much faster than `INSERT INTO` (which reserializes).

**Polaris vending requirement:** all of a table's files must live under its location path. Polaris grants engines access via vended credentials (short-lived S3 creds scoped to that path prefix). So **`add_files` does not move data** — copy the Parquet file into the table's location first, then call the procedure to register it.

In [ ]:
# Copy July data from raw/ into the snapshot table's data directory
# add_files does NOT move data. Files must already be under the table location (Polaris vending requirement)
s3_key_july = 'migration/taxi_snapshot/data/yellow_tripdata_2023-07.parquet'

print("Copying July data into snapshot table location...")
s3_client.copy_object(
    Bucket='warehouse',
    CopySource='warehouse/raw/yellow_tripdata_2023-07.parquet',
    Key=s3_key_july
)
print(f"File copied to s3://warehouse/{s3_key_july}")
print("Original file remains at s3://warehouse/raw/yellow_tripdata_2023-07.parquet")

In [ ]:
# Check current state before adding files
before_count = spark.sql("""
SELECT COUNT(*) as count 
FROM polaris.migration.taxi_from_snapshot
""").collect()[0]['count']

before_files = spark.sql("""
SELECT COUNT(*) as file_count
FROM polaris.migration.taxi_from_snapshot.files
""").collect()[0]['file_count']

print(f"Before add_files:")
print(f"  Rows:  {before_count:,}")
print(f"  Files: {before_files}")

### Using the Add Files Procedure

In [ ]:
# Use add_files to register the new Parquet file without rewriting data
# The parquet.`<path>` syntax is Spark's way of reading raw Parquet files inline
# as if they were a table. We used this same pattern in the CTAS statements above.
print("Adding July data using add_files procedure...")

add_start = time.time()

spark.sql("""
    CALL polaris.system.add_files(
        table => 'polaris.migration.taxi_from_snapshot',
        source_table => 'parquet.`s3a://warehouse/migration/taxi_snapshot/data/yellow_tripdata_2023-07.parquet`'
    )
""").show()

add_time = time.time() - add_start
print(f"add_files completed in {add_time:.2f}s")

In [ ]:
# Verify the addition
after_count = spark.sql("""
SELECT COUNT(*) as count 
FROM polaris.migration.taxi_from_snapshot
""").collect()[0]['count']

after_files = spark.sql("""
SELECT COUNT(*) as file_count
FROM polaris.migration.taxi_from_snapshot.files
""").collect()[0]['file_count']

print(f"After add_files:")
print(f"  Rows:  {after_count:,}  (added {after_count - before_count:,})")
print(f"  Files: {after_files}  (added {after_files - before_files})")

In [ ]:
# View all files now in the snapshot table
spark.sql("""
SELECT 
    file_path,
    ROUND(file_size_in_bytes / 1024 / 1024, 2) as size_mb,
    record_count
FROM polaris.migration.taxi_from_snapshot.files
ORDER BY file_path
""").show(truncate=False)

### View in MinIO

[Open `taxi_snapshot` in MinIO](http://localhost:9001/browser/warehouse/migration/taxi_snapshot/) · `admin / password`

The July file now appears in `data/`; `metadata/` has new manifests referencing it.

### Benchmark: `add_files` vs `INSERT`

Compare `add_files` against `INSERT INTO` for the same July data added to the CTAS table.

In [ ]:
# INSERT the same July data into the CTAS table for comparison
july_df = spark.read.parquet("s3a://warehouse/raw/yellow_tripdata_2023-07.parquet")

print(f"Inserting {july_df.count():,} rows using INSERT INTO...")

insert_start = time.time()
july_df.writeTo("polaris.migration.taxi_from_ctas").append()
insert_time = time.time() - insert_start

print(f"INSERT completed in {insert_time:.2f}s")

print("\n" + "=" * 60)
print("Incremental Load Benchmark: Add Files vs INSERT")
print("=" * 60)
print(f"  add_files time:  {add_time:>8.2f}s  (metadata-only)")
print(f"  INSERT time:     {insert_time:>8.2f}s  (full reserialization)")
print(f"  Speedup:         {insert_time / add_time:>8.1f}x faster with add_files")
print("=" * 60)

### Try It: Migrate Another Month

Add August via both `add_files` and `INSERT INTO` — does the gap hold? Try multiple months at once.

Hint: copy the file into the snapshot table's `data/` dir first (Polaris vending requirement).

In [ ]:
# month = "08"
# raw_file = f"raw/yellow_tripdata_2023-{month}.parquet"
# snapshot_dest = f"migration/taxi_snapshot/data/yellow_tripdata_2023-{month}.parquet"

# Copy the file into the snapshot table location (Polaris vending requirement)
# s3_client.copy_object(Bucket='warehouse', CopySource=f'warehouse/{raw_file}', Key=snapshot_dest)

# Add it using add_files. Time the operation
# add_start = time.time()
# spark.sql(f"""
#     CALL polaris.system.add_files(
#         table => ???,
#         source_table => ???
#     )
# """).show()
# add_aug_time = time.time() - add_start

# INSERT the same data into the CTAS table. Time the operation
# aug_df = spark.read.parquet(f"s3a://warehouse/{raw_file}")
# insert_start = time.time()
# ???
# insert_aug_time = time.time() - insert_start

# Compare results
# print(f"add_files: {add_aug_time:.2f}s  |  INSERT: {insert_aug_time:.2f}s  |  Speedup: {insert_aug_time / add_aug_time:.1f}x")

## View Migration History

Every operation on an Iceberg table creates a snapshot. Let's examine the snapshot history to see our migration steps.

In [ ]:
# View snapshots for the snapshot table
# The summary column is a map of key-value pairs that Iceberg stores for each snapshot.
# Common keys include: added-records, deleted-records, added-data-files,
# deleted-data-files, total-records, total-data-files.
# These keys are consistent across Iceberg engines.
print("Snapshot table history:")
spark.sql("""
SELECT 
    snapshot_id,
    committed_at,
    operation,
    summary['added-records'] as records_added
FROM polaris.migration.taxi_from_snapshot.snapshots
ORDER BY committed_at
""").show(truncate=False)

In [ ]:
# View snapshots for the CTAS table
print("CTAS table history:")
spark.sql("""
SELECT 
    snapshot_id,
    committed_at,
    operation,
    summary['added-records'] as records_added
FROM polaris.migration.taxi_from_ctas.snapshots
ORDER BY committed_at
""").show(truncate=False)

## Cleanup

In [ ]:
# Optional: Drop the tables if you want to start fresh
# spark.sql("DROP TABLE IF EXISTS polaris.migration.customers_from_csv")
# spark.sql("DROP TABLE IF EXISTS polaris.migration.taxi_from_ctas")
# spark.sql("DROP TABLE IF EXISTS polaris.migration.taxi_from_snapshot")
# print("Tables dropped!")